# 결정 트리 복습 워크북 (빈칸 채우기)

각 셀의 `____` 빈칸을 채우고 실행하라. 개념 빈칸은 머릿속으로 답을 정한 뒤 토글로 확인하고, 코드 빈칸은 채워서 직접 실행해 결과를 확인한다. 정답은 각 문제 아래 `▶ 정답 보기`에 있다.

> 앞 셀에서 정의한 변수(`gini`, `entropy`, `X`, `y` 등)를 뒤 셀이 사용하므로 **위에서 아래로 순서대로** 실행한다.


## 1. 개념 빈칸

다음 문장의 빈칸에 들어갈 말을 떠올려 보라.

(1) 한 노드에 한 클래스만 있으면 ____ 하다고 하며, 이때 지니와 엔트로피는 모두 ____ 이다.

(2) 두 클래스가 반반으로 섞일 때 불순도는 ____ 가 된다.

(3) 결정 트리는 모든 변수와 분할점 중 ____ 이 가장 큰 것을 골라 가지를 친다.

(4) 변수 중요도가 0인 변수는 트리가 분할에 ____ 사용한 변수다.

<details><summary>▶ 정답 보기</summary>

(1) 순수(pure), 0
(2) 최대(가장 큼)
(3) 정보 이득(information gain)
(4) 한 번도 사용하지 않은(미사용)

</details>

## 2. 환경 준비

이 셀은 그대로 실행한다.

In [ ]:
try:
    import koreanize_matplotlib
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "koreanize-matplotlib"])
    import koreanize_matplotlib
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['figure.dpi'] = 110

## 3. 지니 지수 함수 완성

$G = 1 - \sum_k p_k^2$ 를 구현하라. 비율 `p`를 제곱해 합한 값을 1에서 뺀다.

In [ ]:
def gini(counts):
    counts = np.array(counts, dtype=float)
    p = counts / counts.____()          # 각 클래스 비율
    return 1.0 - np.sum(p ** ____)

print(gini([10, 0, 0]), gini([5, 5, 0]), gini([4, 3, 3]))

<details><summary>▶ 정답 보기</summary>

```python
def gini(counts):
    counts = np.array(counts, dtype=float)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)

print(gini([10, 0, 0]), gini([5, 5, 0]), gini([4, 3, 3]))
```

`counts.sum()`으로 전체 개수를 나눠 비율을 구하고, 비율의 제곱합을 1에서 뺀다. 결과는 0.0, 0.5, 0.653 부근이다.

</details>

## 4. 엔트로피 함수 완성

$H = -\sum_k p_k \log_2 p_k$ 를 구현하라. 0인 비율은 로그가 정의되지 않으므로 미리 걸러 낸다.

In [ ]:
def entropy(counts):
    counts = np.array(counts, dtype=float)
    p = counts / counts.sum()
    p = p[p > 0]                        # log(0) 방지
    return -np.sum(p * np.____(p))      # 밑이 2인 로그

print(round(entropy([10, 0, 0]), 3), round(entropy([10, 10, 10]), 3))

<details><summary>▶ 정답 보기</summary>

```python
def entropy(counts):
    counts = np.array(counts, dtype=float)
    p = counts / counts.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print(round(entropy([10, 0, 0]), 3), round(entropy([10, 10, 10]), 3))
```

`np.log2`가 밑이 2인 로그다. 순수 노드는 0.0, 3클래스 균등은 약 1.585($\log_2 3$)이다.

</details>

## 5. 와인 데이터 불러오기

이 셀은 그대로 실행한다.

In [ ]:
try:
    from sklearn.datasets import load_wine
    wine = load_wine()
    X, y = wine.data, wine.target
    feature_names = list(wine.feature_names)
    class_names = [f"품종{i}" for i in range(len(wine.target_names))]
except Exception:
    rng = np.random.default_rng(42)
    X = rng.normal(0, 1, (178, 13)); y = rng.integers(0, 3, 178)
    feature_names = [f"x{i}" for i in range(13)]; class_names = ['품종0','품종1','품종2']
n = len(y)
print("샘플 수:", n, "| 변수 수:", len(feature_names))

## 6. 루트 노드 불순도

`np.bincount(y)`로 세 품종의 개수를 세고, 루트의 지니와 엔트로피를 구하라.

In [ ]:
counts = np.____(y)                    # 클래스별 개수
print("루트 지니   :", round(gini(counts), 4))
print("루트 엔트로피:", round(____(counts), 4))

<details><summary>▶ 정답 보기</summary>

```python
counts = np.bincount(y)
print("루트 지니   :", round(gini(counts), 4))
print("루트 엔트로피:", round(entropy(counts), 4))
```

루트 지니는 약 0.658, 엔트로피는 약 1.567이다.

</details>

## 7. 정보 이득 계산

`flavanoids`를 1.575에서 자를 때의 정보 이득을 구하라. 정보 이득은 부모 엔트로피에서 자식들의 **가중 평균** 엔트로피를 뺀 값이다.

In [ ]:
j = feature_names.index('flavanoids')
mask = X[:, j] <= 1.575
left_y, right_y = y[mask], y[~mask]

H_parent = entropy(np.bincount(y, minlength=3))
H_children = (len(left_y)/n  * entropy(np.bincount(left_y,  minlength=3))
            + len(right_y)/n * entropy(np.bincount(right_y, minlength=3)))
info_gain = H_parent ____ H_children    # 빼기

print("정보 이득:", round(info_gain, 4))

<details><summary>▶ 정답 보기</summary>

```python
info_gain = H_parent - H_children
print("정보 이득:", round(info_gain, 4))
```

부모 엔트로피에서 자식 가중 평균 엔트로피를 빼면 정보 이득이며, 이 분할의 값은 약 0.7 이상으로 매우 크다. 그래서 트리가 `flavanoids`를 루트로 고른다.

</details>

## 8. 결정 트리 학습

엔트로피 기준, 깊이 3인 트리를 학습하라.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

clf = DecisionTreeClassifier(criterion='____', max_depth=____, random_state=42)
clf.fit(X, y)
print("학습 정확도:", round(clf.score(X, y), 4))

<details><summary>▶ 정답 보기</summary>

```python
clf = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
clf.fit(X, y)
print("학습 정확도:", round(clf.score(X, y), 4))
```

`criterion='entropy'`로 엔트로피 기준을, `max_depth=3`으로 깊이를 제한한다. 정확도는 약 0.99다.

</details>

## 9. 트리 시각화

학습된 트리를 `plot_tree`로 그려라. 각 노드에 색을 채우려면 `filled=True`를 준다.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
plot_tree(clf, feature_names=feature_names, class_names=class_names,
          filled=____, rounded=True, fontsize=9, ax=ax)
plt.tight_layout()
plt.show()

<details><summary>▶ 정답 보기</summary>

```python
fig, ax = plt.subplots(figsize=(16, 9))
plot_tree(clf, feature_names=feature_names, class_names=class_names,
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.tight_layout()
plt.show()
```

`filled=True`면 노드의 우세 클래스에 따라 색이 칠해져 순수도를 한눈에 볼 수 있다.

</details>

## 10. 변수 중요도

깊이 제한 없는 트리를 학습하고, `feature_importances_`로 변수 중요도를 막대그래프로 그려라.

In [ ]:
clf_full = DecisionTreeClassifier(criterion='entropy', random_state=42).fit(X, y)
importance = clf_full.____               # 변수 중요도 배열
order = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh([feature_names[i] for i in order], importance[order], color='#1F3A5F')
ax.set_title('변수 중요도')
plt.tight_layout()
plt.show()

<details><summary>▶ 정답 보기</summary>

```python
clf_full = DecisionTreeClassifier(criterion='entropy', random_state=42).fit(X, y)
importance = clf_full.feature_importances_
order = np.argsort(importance)
...
```

`feature_importances_` 속성에 각 변수가 줄여 준 불순도의 합이 정규화되어 들어 있다. `flavanoids`, `proline`, `color_intensity`가 상위다.

</details>

## 11. 지니와 엔트로피 비교

두 기준으로 깊이 3 트리를 각각 학습하고 5겹 교차검증 정확도를 비교하라.

In [ ]:
from sklearn.model_selection import cross_val_score

for crit in ['gini', '____']:           # 두 기준
    t = DecisionTreeClassifier(criterion=crit, max_depth=3, random_state=42)
    cv = cross_val_score(t, X, y, cv=____).mean()   # 5겹
    print(f"{crit:8s} → 5겹 CV 정확도 {cv:.4f}")

<details><summary>▶ 정답 보기</summary>

```python
for crit in ['gini', 'entropy']:
    t = DecisionTreeClassifier(criterion=crit, max_depth=3, random_state=42)
    cv = cross_val_score(t, X, y, cv=5).mean()
    print(f"{crit:8s} → 5겹 CV 정확도 {cv:.4f}")
```

두 기준의 교차검증 정확도는 거의 같다. 그래서 계산이 가벼운 지니가 기본값으로 쓰인다.

</details>

---

## 마무리 점검

스스로 답해 보라. 토글로 확인한다.

1. 트리의 루트가 `flavanoids`를 첫 질문으로 고른 이유는?
2. 잎 노드의 엔트로피가 0에 가까운 것은 무엇을 뜻하는가?
3. `max_depth`를 너무 크게 하면 생기는 문제는?

<details><summary>▶ 정답 보기</summary>

1. 모든 변수의 최선의 분할 중 `flavanoids`의 분할이 정보 이득이 가장 컸기 때문이다.
2. 그 노드가 거의 한 품종으로만 채워진 순수한 상태라는 뜻이다.
3. 학습 데이터의 잡음까지 외워 과적합이 일어나며, 새 데이터에 대한 일반화 성능이 떨어진다.

</details>